In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
train_base_df = pd.read_csv('../data/csv_files/train/train_base.csv')
train_static_0_0_df = pd.read_csv('../data/csv_files/train/train_static_0_0.csv')
train_static_0_1_df = pd.read_csv('../data/csv_files/train/train_static_0_1.csv')
train_static_cb_0_df = pd.read_csv('../data/csv_files/train/train_static_cb_0.csv')
train_person_1_df = pd.read_csv('../data/csv_files/train/train_person_1.csv')
train_credit_bureau_a_1_0_df = pd.read_csv('../data/csv_files/train/train_credit_bureau_a_1_0.csv')
train_credit_bureau_a_1_1_df = pd.read_csv('../data/csv_files/train/train_credit_bureau_a_1_1.csv')
train_credit_bureau_a_2_df = pd.read_csv('../data/csv_files/train/train_credit_bureau_a_2.csv')
train_applprev_1_df = pd.read_csv('../data/csv_files/train/train_applprev_1.csv')
train_tax_registry_a_1_df = pd.read_csv('../data/csv_files/train/train_tax_registry_a_1.csv')

In [ ]:
# Concatenate the two credit bureau dataframes into one
train_credit_bureau_a_1_df = pd.concat([train_credit_bureau_a_1_0_df, train_credit_bureau_a_1_1_df],
                                       axis=0, ignore_index=True)
# Concatenate the two static dataframes into one
train_static_0_df = pd.concat([train_static_0_0_df, train_static_0_1_df],
                                       axis=0, ignore_index=True)

In [ ]:
# Sort so num_group1 == 0 comes first for every case_id
train_person_1_df = train_person_1_df.sort_values(by=["case_id", "num_group1"])

# Aggregate person-level data to case-level data
person_agg = train_person_1_df.groupby("case_id").agg(
    num_applicants=("num_group1", "count"),
    max_income=("mainoccupationinc_384A", "max"),
    sum_income=("mainoccupationinc_384A", "sum"),
    primary_housing_type=("housingtype_772M", "first")
).reset_index()

# Aggregate credit bureau features per applicant
cb1_agg = train_credit_bureau_a_1_df.groupby("case_id").agg(
    # Thin-file signal (0 = Thin file, 1+ = Established)
    bureau_rowcount=("num_group1", "count"),

    # Overdue Amount: Both peak and average
    max_overdue_amount=("overdueamountmax_950A", "max"),
    mean_overdue_amount=("overdueamountmax_950A", "mean"),

    # Days Past Due: Peak severity and average discipline
    max_days_past_due=("pmts_dpdvalue_108P", "max"),
    mean_days_past_due=("pmts_dpdvalue_108P", "mean"),

    # Credit Amounts: Average capacity and peak exposure
    mean_past_credit_limit=("credamount_770A", "mean"),
    max_past_credit_limit=("credamount_770A", "max")
).reset_index()

# Aggregate bureau features for the monthly data
cb2_agg = train_credit_bureau_a_2_df.groupby("case_id").agg(
    # Record Count
    total_monthly_records=("pmts_month_158T", "count"),

    # Overdue Amount Aggregations (Peak, Average, and Cumulative)
    max_monthly_overdue=("pmts_overdue_1140A", "max"),
    mean_monthly_overdue=("pmts_overdue_1140A", "mean"),
    sum_monthly_overdue=("pmts_overdue_1140A", "sum")
).reset_index()

train_applprev_1_df["approvaldate_319D"] = pd.to_datetime(train_applprev_1_df["approvaldate_319D"]
                                                          , errors="coerce")

# Aggregate prior application features
applprev_agg = train_applprev_1_df.groupby("case_id").agg(

    prior_app_count=("num_group1", "count"),

    # Requested Amount signals (peak, average, and most recent)
    max_prior_requested_amt=("credamount_590A", "max"),
    mean_prior_requested_amt=("credamount_590A", "mean"),

    # Recency Signal
    most_recent_approval_date=("approvaldate_319D", "max"),
).reset_index()

# Aggregate tax registry features per applicant
tax_agg = train_tax_registry_a_1_df.groupby("case_id").agg(
    tax_filing_count=("num_group1", "count"),
    mean_tax_income=("amount_4527230A", "mean"),
    max_tax_income=("amount_4527230A", "max"),
    num_unique_employers=("name_4527232M", "nunique")
).reset_index()

In [7]:
# Create a master dataframe to hold all features
df_master = train_base_df.copy()

# Merge all aggregated features into the master dataframe
df_master = df_master.merge(train_static_0_df, on="case_id", how="left")
df_master = df_master.merge(train_static_cb_0_df, on="case_id", how="left")
df_master = df_master.merge(person_agg, on="case_id", how="left")
df_master = df_master.merge(cb1_agg, on="case_id", how="left")
df_master = df_master.merge(cb2_agg, on="case_id", how="left")
df_master = df_master.merge(applprev_agg, on="case_id", how="left")
df_master = df_master.merge(tax_agg, on="case_id", how="left")

In [8]:
# Fill NaN values for numeric columns with 0 to extract thin-file segment flag
df_master["bureau_rowcount"] = df_master["bureau_rowcount"].fillna(0)
df_master["total_monthly_records"] = df_master["total_monthly_records"].fillna(0)
df_master["prior_app_count"] = df_master["prior_app_count"].fillna(0)
df_master["tax_filing_count"] = df_master["tax_filing_count"].fillna(0)
# Thin-File Segment Flag
df_master["is_thin_file"] = (df_master["bureau_rowcount"] == 0).astype(int)

In [9]:
df_master.head(10)

,case_id,date_decision,MONTH,WEEK_NUM,target,mainoccupationinc_384A,credamount_770A,annuity_780A,days_employed_700P,education_927M,...,sum_monthly_overdue,prior_app_count,max_prior_requested_amt,mean_prior_requested_amt,most_recent_approval_date,tax_filing_count,mean_tax_income,max_tax_income,num_unique_employers,is_thin_file
0,1,2020-04-28,202004,69,0,28314.0,16373.0,600.0,1976.0,6def22f0,...,NaN,1.0,9636.0,9636.000000,2017-11-11,3.0,24916.333333,27314.0,3.0,1
1,2,2019-08-13,201908,32,0,25386.0,13707.0,725.0,NaN,10795bac,...,49.0,4.0,29168.0,14859.750000,2016-10-13,2.0,17402.000000,20724.0,2.0,0
2,3,2019-01-01,201901,0,1,57416.0,12676.0,619.0,762.0,242b264b,...,NaN,2.0,28168.0,18844.500000,2014-06-18,0.0,NaN,NaN,NaN,1
3,4,2019-08-27,201908,34,1,31763.0,44528.0,830.0,355.0,6def22f0,...,0.0,2.0,17924.0,12384.500000,2017-03-15,2.0,20431.000000,25348.0,2.0,0
4,5,2019-03-26,201903,12,0,11990.0,21065.0,1282.0,0.0,6def22f0,...,140.0,0.0,NaN,NaN,NaT,2.0,28010.000000,34499.0,2.0,0
5,6,2019-12-31,201912,52,0,12166.0,9069.0,1071.0,1424.0,242b264b,...,405.0,3.0,17161.0,10021.000000,2017-11-09,1.0,17035.000000,17035.0,1.0,0
6,7,2019-10-22,201910,42,0,NaN,16415.0,NaN,NaN,6def22f0,...,339.0,3.0,7782.0,5773.666667,2017-01-12,2.0,25407.000000,33059.0,2.0,0
7,8,2020-04-21,202004,68,0,15939.0,18323.0,NaN,NaN,6def22f0,...,225.0,0.0,NaN,NaN,NaT,0.0,NaN,NaN,NaN,0
8,9,2019-11-19,201911,46,1,13755.0,11738.0,1228.0,646.0,2d4a2bc4,...,47.0,0.0,NaN,NaN,NaT,2.0,18018.500000,27136.0,2.0,0
9,10,2019-04-23,201904,16,0,17947.0,19116.0,750.0,493.0,10795bac,...,1245.0,0.0,NaN,NaN,NaT,3.0,19170.000000,23172.0,2.0,0


In [10]:
# Missing value count and percentage for every column in df_master
missing_df = pd.DataFrame({
    "missing_count": df_master.isnull().sum(),
    "pct_missing": (df_master.isnull().mean() * 100).round(2)
})

print(missing_df)

                           missing_count  pct_missing
case_id                                0         0.00
date_decision                          0         0.00
MONTH                                  0         0.00
WEEK_NUM                               0         0.00
target                                 0         0.00
mainoccupationinc_384A             49651         4.97
credamount_770A                    19959         2.00
annuity_780A                       99677         9.97
days_employed_700P                150571        15.06
education_927M                         0         0.00
maritalstatus_703M                     0         0.00
birth_259D                             0         0.00
riskassesment_940T                334965        33.50
numberofqueries_146L              299821        29.98
description_5085714M              299821        29.98
num_applicants                         0         0.00
max_income                         26178         2.62
sum_income                  

In [11]:
# percentage of thin-file applicants
df_master["is_thin_file"].mean() * 100

np.float64(29.9821)

In [12]:
# Check the missing columns that are either int or float
missing_cols = missing_df[(missing_df["missing_count"] != 0) & (df_master.dtypes.isin(["int64", "float64"]))].index.tolist()
missing_cols

[]

In [13]:
# percentage of all rows in df_master that satisfy both conditions:
# is_thin_file == 1
# col is null
for col in missing_cols:
    percentage = (((df_master["is_thin_file"] == 1)& df_master[col].isnull()).mean()* 100)
    print(f"{col}: {percentage:.2f}%")

In [14]:
# Bureau and applprev cols
bureau_applprev_cols = list(cb1_agg) + list(cb2_agg) + list(train_static_cb_0_df) + list(applprev_agg)
bureau_applprev_cols

['case_id',
 'bureau_rowcount',
 'max_overdue_amount',
 'mean_overdue_amount',
 'max_days_past_due',
 'mean_days_past_due',
 'mean_past_credit_limit',
 'max_past_credit_limit',
 'case_id',
 'total_monthly_records',
 'max_monthly_overdue',
 'mean_monthly_overdue',
 'sum_monthly_overdue',
 'case_id',
 'riskassesment_940T',
 'numberofqueries_146L',
 'description_5085714M',
 'case_id',
 'prior_app_count',
 'max_prior_requested_amt',
 'mean_prior_requested_amt',
 'most_recent_approval_date']

In [15]:
for col in missing_cols:
    if col in bureau_applprev_cols:
        # Calculate mean before adding thin-file zeros
        mean_value = df_master.loc[df_master["is_thin_file"] == 0, col].mean()

        # Thin-file applicants: missing -> 0
        df_master.loc[df_master["is_thin_file"] == 1, col] = df_master.loc[df_master["is_thin_file"] == 1, col].fillna(0)

        # Non-thin-file applicants: missing -> mean
        df_master[col] = df_master[col].fillna(mean_value)

In [16]:
# fillna using mean -- cols have na percentage <16%
cols_mean = ['mainoccupationinc_384A','credamount_770A', 'annuity_780A', 'days_employed_700P', 'max_income']
for col in cols_mean:
    # Calculate mean
    mean_value = df_master.loc[df_master["is_thin_file"] == 0, col].mean()
    # missing → mean
    df_master[col] = df_master[col].fillna(mean_value)

In [17]:
# Drop cols of types that aren't int or float with missing values
df_master.drop(columns=['description_5085714M', 'most_recent_approval_date'], inplace = True)

In [18]:
# Drop tax info due to redundancy --> cross-check on the income the applicant makes
df_master.drop(columns=['max_tax_income', 'mean_tax_income', 'num_unique_employers'], inplace = True)

In [19]:
df_master["date_decision"] = pd.to_datetime(df_master["date_decision"])
df_master["birth_259D"] = pd.to_datetime(df_master["birth_259D"])
# Calculate the Age in years
df_master["age"] = ((df_master["date_decision"] - df_master["birth_259D"]).dt.days / 365.25)

In [20]:
missing_df = pd.DataFrame({
    "missing_count": df_master.isnull().sum(),
    "pct_missing": (df_master.isnull().mean() * 100).round(2)
})

print(missing_df)


                          missing_count  pct_missing
case_id                               0         0.00
date_decision                         0         0.00
MONTH                                 0         0.00
WEEK_NUM                              0         0.00
target                                0         0.00
mainoccupationinc_384A                0         0.00
credamount_770A                       0         0.00
annuity_780A                          0         0.00
days_employed_700P                    0         0.00
education_927M                        0         0.00
maritalstatus_703M                    0         0.00
birth_259D                            0         0.00
riskassesment_940T               334965        33.50
numberofqueries_146L             299821        29.98
num_applicants                        0         0.00
max_income                            0         0.00
sum_income                            0         0.00
primary_housing_type                  0       

In [21]:
# Identify columns ending with 'M' (masked categorical features)
masked_cols = [col for col in df_master.columns if col.endswith('M')]
print("Masked categorical columns found:", masked_cols)

# Display the unique values (distinct values) for each identified masked column
for col in masked_cols:
    unique_vals = df_master[col].dropna().unique()
    print(f"\nDistinct values for column '{col}' (Count: {len(unique_vals)}):")
    if len(unique_vals) <= 20:
        print(unique_vals)
    else:
        print(unique_vals[:20], "... [truncated]")

Masked categorical columns found: ['WEEK_NUM', 'education_927M', 'maritalstatus_703M']

Distinct values for column 'WEEK_NUM' (Count: 92):
[69 32  0 34 12 52 42 68 46 16 41 53 18 49  8 55 24 62  1 33] ... [truncated]

Distinct values for column 'education_927M' (Count: 5):
['6def22f0' '10795bac' '242b264b' '2d4a2bc4' 'f937ecaa']

Distinct values for column 'maritalstatus_703M' (Count: 4):
['f62a4fc7' 'ea6fb4b5' 'e78cb9c0' 'b58630a7']


In [22]:
# Perform One-Hot Encoding on education_927M and maritalstatus_703M
df_master = pd.get_dummies(
    df_master,
    columns=['education_927M', 'maritalstatus_703M'],
    prefix=['edu', 'marital'],
    dtype=int
)

# Display the new columns to confirm they are generated as binary integers
new_encoded_cols = [col for col in df_master.columns if col.startswith(('edu_', 'marital_'))]
print("Newly created One-Hot Encoded columns:")
print(new_encoded_cols)

# Display preview
display(df_master[['case_id'] + new_encoded_cols].head())

Newly created One-Hot Encoded columns:
['edu_10795bac', 'edu_242b264b', 'edu_2d4a2bc4', 'edu_6def22f0', 'edu_f937ecaa', 'marital_b58630a7', 'marital_e78cb9c0', 'marital_ea6fb4b5', 'marital_f62a4fc7']


,case_id,edu_10795bac,edu_242b264b,edu_2d4a2bc4,edu_6def22f0,edu_f937ecaa,marital_b58630a7,marital_e78cb9c0,marital_ea6fb4b5,marital_f62a4fc7
0,1,0,0,0,1,0,0,0,0,1
1,2,1,0,0,0,0,0,0,1,0
2,3,0,1,0,0,0,0,1,0,0
3,4,0,0,0,1,0,0,1,0,0
4,5,0,0,0,1,0,0,0,0,1
